---
# RotNet Implementation 
---

In this exercise, we will implement a self-supervised learning method called **RotNet** and apply it to the CIFAR-10 dataset.

You do not need to read the paper to complete the exercise, but it gives useful background on why predicting image rotations can lead to useful visual features.  
Paper: [Unsupervised Representation Learning by Predicting Image Rotations](https://arxiv.org/abs/1803.07728)

---
# Imports
---

Run the code cell below after you have set up everything correctly (either locally or in Colab).

In [10]:
import importlib
import torch
import data
import downstream
import models
import pretrain
import tests
import visual

---
## Image Rotations
---

In supervised learning, labels usually come from the dataset. In RotNet, we create labels ourselves.

For each image, we randomly choose one of four rotations:

| Label | Rotation |
|---:|---:|
| `0` | 0 degrees |
| `1` | 90 degrees |
| `2` | 180 degrees |
| `3` | 270 degrees |

The model receives the rotated image and has to predict the rotation label.

This gives us a self-supervised pretraining task: we can train on images without using their class labels.

## **Task:**

Implement all rotation functions in `pretrain.py`:

- `apply_rotations` receives a batch of images and a tensor of rotation labels. It should return a new batch where each image was rotated according to its label.
- `make_random_rotations` is used during training. It should sample random rotation labels and apply the corresponding rotations.
- `make_deterministic_rotations` is used during evaluation. It should create a fixed pattern of labels so that the evaluation accuracy is stable across runs.

**Hints:**

- Use `torch.rot90` to rotate the spatial image dimensions.
- **Avoid rotating images one by one in a Python loop. This can make training very slow.**  
  A more efficient approach is to loop over the four possible rotation labels and use boolean masks to rotate all images with the same label at once.
- `make_random_rotations` can use `torch.randint`.
- `make_deterministic_rotations` can use `torch.arange` and the modulo operator `%`.
- `apply_rotations` should not modify the input tensor in-place.

In [11]:
importlib.reload(pretrain)
tests.test_rotations()

✅ PASS: rotation functions are correct.


True

**Important**: The test checks correctness, not speed. If training is very slow, check whether `apply_rotations` rotates images one by one in a Python loop.  
A faster approach is to rotate all images with the same label together using boolean masks.

---
## Visualizing the Rotations
---

Before training a model, let us visualize the training data.  
Note that there will be random black borders, since we also apply random cropping transformations.

In [12]:
importlib.reload(pretrain)

train_loader, _ = data.make_cifar10_loaders(batch_size=8, num_workers=0)

x, _ = next(iter(train_loader))
rotated_x, rotation_labels = pretrain.make_deterministic_rotations(x)

visual.rotation_examples(rotated_x, rotation_labels).show()

---
## Rotation Classifier
---

The RotNet model consists of two parts:

1. a CNN encoder that extracts image features,
2. a classification head that predicts the rotation label.

The CNN encoder is already provided. You only need to implement the rotation classifier that combines the encoder with a prediction head.

## **Task:**

Implement `RotationClassifier` in `models.py`.

The model should predict one of four rotation classes: 0, 90, 180, or 270 degrees.  
Use two linear layers in the head, with a hidden layer of size `128`.

**Hints:**

- Create the encoder inside the constructor using `CNNEncoder()`.
- Store the encoder as `self.encoder`.
- Store the prediction head as `self.head`.
- The model should return raw logits, not probabilities, so do not apply `softmax`.

In [13]:
importlib.reload(models)
tests.test_rotation_classifier()

✅ PASS: RotationClassifier is correct.


True

---
## RotNet Training Step
---

We now implement one training step for the rotation prediction task.

For each mini-batch, the model should:

1. create randomly rotated images,
2. predict the rotation labels,
3. compute the loss.

## **Task:**

Implement `rotnet_training_step` in `pretrain.py`.

**Hint:** Move the image batch to the selected device before applying rotations.

In [14]:
importlib.reload(pretrain)
tests.test_rotnet_training_step()

✅ PASS: RotNet training step is correct.


True

---
## Training Setup
---

The code below uses `num_workers=2` as a conservative default. This should work on most machines.  
If loading data is slow, you can try increasing it to `4`. If you get dataloader errors, set it to `0`.

In [15]:
num_workers = 2
batch_size = 256

In [16]:
device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "mps"
    if torch.backends.mps.is_available() and torch.backends.mps.is_built() else "cpu")

print(f"Using device: {device}")

Using device: cuda:0


---
## RotNet Pretraining
---

We can now train the rotation classifier.

During this phase, both the encoder and the rotation prediction head are trained. The original CIFAR-10 labels are ignored.

With the provided architecture and hyperparameters, the rotation classifier should reach at least **80% test accuracy after 30 epochs**.

In [18]:
importlib.reload(models)
importlib.reload(pretrain)

# Adjust these if you want to try to find better hyperparameters.
pretrain_lr = 1e-3
pretrain_weight_decay = 1e-4
pretrain_epochs = 40

rotnet, train_loss, test_acc = pretrain.train_rotnet(
    batch_size=batch_size,
    lr=pretrain_lr,
    weight_decay=pretrain_weight_decay,
    epochs=pretrain_epochs,
    device=device,
    num_workers=num_workers,
)
visual.show_training_stats(train_loss, test_acc, title="RotNet Pretraining").show()

Train loss: 0.762, Test accuracy: 67.34 %:  18%|█▊        | 7/40 [02:49<13:16, 24.13s/it]Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x785949d9a700>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Train loss: 0.735, Test accuracy: 71.13 %:  20%|██        | 8/40 [03:13<12:53, 24.19s/it]Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x785949d9a700>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
  


Final test accuracy: 82.63 %


---
## Downstream Classifier
---

We now replace the rotation prediction head with a CIFAR-10 classification head.

The encoder is reused as a fixed feature extractor.

## **Task:**

Implement `DownstreamClassifier` in `models.py`.

The class should combine a pretrained encoder with a new classification head for CIFAR-10.  
The classification head should be a single linear layer with 10 output units.

The encoder should stay fixed during downstream training. Only the new head should be trained.

**Hints:**

- The encoder output dimension is stored in `encoder.out_dim`.
- Avoid computing gradients through the encoder. You know how to disable gradients.
- Put the encoder in evaluation mode before computing features.
- The model should return raw logits, not probabilities.

In [ ]:
importlib.reload(models)
tests.test_downstream_classifier()

✅ PASS: DownstreamClassifier is correct.


True

---
## Downstream Optimizer
---

The downstream model contains both the pretrained encoder and the new classification head.

For this part, we only want to optimize the new head.

## **Task:**

Implement `create_downstream_model_and_optimizer` in `downstream.py`.

The function receives the trained rotation classifier and should return:

1. a downstream classifier,
2. an optimizer for downstream training.

**Hints:**

- The trained rotation classifier contains the pretrained encoder.
- Check carefully which parameters should be passed to the optimizer.
- Do not pass all model parameters to the optimizer.

In [ ]:
importlib.reload(downstream)
tests.test_create_downstream_model_and_optimizer()

✅ PASS: downstream model and optimizer are created correctly.


True

---
## Downstream Training
---

We can now train the downstream classifier and evaluate it on CIFAR-10.

With the provided architecture and hyperparameters, the downstream classifier should reach about **57–60% test accuracy after 20 epochs**. When you reach significantly higher test accuracy, then you probably jointly trained the CNN encoder.

In [ ]:
importlib.reload(models)
importlib.reload(downstream)

# Adjust these if you want to try to find better hyperparameters.
downstream_lr = 1e-2
downstream_weight_decay = 1e-4
downstream_epochs = 20

model, train_loss, test_acc = downstream.train_downstream(
    rotnet=rotnet,
    batch_size=batch_size,
    lr=downstream_lr,
    weight_decay=downstream_weight_decay,
    epochs=downstream_epochs,
    device=device,
    num_workers=num_workers,
)
visual.show_training_stats(train_loss, test_acc, title="Downstream Training").show()